# 🔤 Brahmi Script OCR — TrOCR Training on Google Colab

This notebook trains the TrOCR model on a synthetic Brahmi dataset using Colab's **free GPU**.

**Pipeline:**
1. Mount Google Drive & upload project files
2. Install dependencies
3. Generate synthetic Brahmi dataset
4. Fine-tune `microsoft/trocr-small-printed`
5. Save trained model to Google Drive

---

> ⚠️ **Before running:** Go to **Runtime → Change runtime type → GPU (T4)**

## Step 0 — Verify GPU is available

In [ ]:
import torch
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device      : {torch.cuda.get_device_name(0)}")
    print(f"GPU memory      : {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("❌ No GPU detected! Go to Runtime → Change runtime type → GPU")

## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2 — Upload project to Colab

**Option A (recommended):** If you pushed your code to GitHub:
```
!git clone https://github.com/YOUR_USERNAME/brahmi_ocr_project.git /content/brahmi_ocr_project
```

**Option B:** Upload from Google Drive (copy your project folder to Drive first):
```
!cp -r /content/drive/MyDrive/brahmi_ocr_project /content/brahmi_ocr_project
```

**Option C:** Upload the zip directly (upload `brahmi_ocr_project.zip` using the Files panel on the left).

In [ ]:
# ============================================================
# CHOOSE ONE OPTION AND UNCOMMENT IT:
# ============================================================

# --- Option A: Clone from GitHub ---
# !git clone https://github.com/YOUR_USERNAME/brahmi_ocr_project.git /content/brahmi_ocr_project

# --- Option B: Copy from Google Drive ---
# !cp -r "/content/drive/MyDrive/brahmi_ocr_project" /content/brahmi_ocr_project

# --- Option C: Upload zip and unzip ---
# from google.colab import files
# uploaded = files.upload()  # upload brahmi_ocr_project.zip
# !unzip -q brahmi_ocr_project.zip -d /content/

# --- Verify ---
import os
PROJECT_DIR = '/content/brahmi_ocr_project'
assert os.path.exists(PROJECT_DIR), f"Project not found at {PROJECT_DIR}!"
print("✅ Project found!")
!ls {PROJECT_DIR}

## Step 3 — Install dependencies

In [ ]:
%cd /content/brahmi_ocr_project
!pip install -q -r requirements.txt

## Step 4 — Generate synthetic Brahmi dataset

Generates **20,000 images** (5k chars + 10k words + 5k phrases) with augmentations.

This takes ~5–10 minutes.

In [ ]:
# Check if dataset already exists (skip if re-running)
labels_path = os.path.join(PROJECT_DIR, 'dataset', 'labels.txt')
if os.path.exists(labels_path):
    num_existing = sum(1 for _ in open(labels_path))
    print(f"Dataset already exists with {num_existing} samples.")
    if num_existing >= 20000:
        print("Skipping generation. Delete dataset/labels.txt to regenerate.")
    else:
        print("Regenerating full dataset...")
        !python dataset/generate_synthetic.py --num_chars 5000 --num_words 10000 --num_phrases 5000
else:
    !python dataset/generate_synthetic.py --num_chars 5000 --num_words 10000 --num_phrases 5000

In [ ]:
# Quick check: show some labels and a sample image
!head -5 dataset/labels.txt
print(f"\nTotal images: {len(os.listdir('dataset/images'))}")
print(f"Train split : {sum(1 for _ in open('dataset/splits/train.txt'))}")
print(f"Val split   : {sum(1 for _ in open('dataset/splits/val.txt'))}")
print(f"Test split  : {sum(1 for _ in open('dataset/splits/test.txt'))}")

In [ ]:
# Display a sample image
from PIL import Image
import matplotlib.pyplot as plt

sample_imgs = sorted(os.listdir('dataset/images'))[:4]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, fname in zip(axes, sample_imgs):
    img = Image.open(f'dataset/images/{fname}')
    ax.imshow(img)
    ax.set_title(fname, fontsize=9)
    ax.axis('off')
plt.suptitle('Sample Generated Brahmi Images', fontsize=14)
plt.tight_layout()
plt.show()

## Step 5 — Train the TrOCR model 🚀

Fine-tunes `microsoft/trocr-small-printed` on the Brahmi dataset.

| Setting | Value |
|---------|-------|
| Base model | `microsoft/trocr-small-printed` |
| Epochs | 10 |
| Batch size | 8 (Colab GPU can handle more than local CPU) |
| Learning rate | 5e-5 |
| FP16 | Auto (enabled on GPU) |

**Estimated time:** ~30–60 min on T4 GPU

In [ ]:
!python training/train.py \
    --epochs 10 \
    --batch_size 8 \
    --lr 5e-5 \
    --data_dir dataset \
    --output_dir model/brahmi_trocr

## Step 6 — Test inference on a sample image

In [ ]:
# Pick a test image
test_images = open('dataset/splits/test.txt').read().strip().split('\n')
test_img = test_images[0]
print(f"Test image: {test_img}")

# Get ground truth
labels = {}
for line in open('dataset/labels.txt', encoding='utf-8'):
    parts = line.strip().split('\t')
    if len(parts) == 2:
        labels[parts[0]] = parts[1]
print(f"Ground truth: {labels.get(test_img, 'N/A')}")

# Run inference
!python inference/predict.py --image dataset/images/{test_img} --model_dir model/brahmi_trocr

In [ ]:
# Batch test: run inference on 5 test images and compare
import sys
sys.path.insert(0, '/content/brahmi_ocr_project')

from inference.predict import load_trained_model, predict

processor, model, device = load_trained_model('model/brahmi_trocr')
print(f"Model loaded on: {device}\n")

correct = 0
total = min(5, len(test_images))
for img_name in test_images[:total]:
    img_path = f'dataset/images/{img_name}'
    pred = predict(img_path, processor, model, device)
    gt = labels.get(img_name, '')
    match = '✅' if pred.strip() == gt.strip() else '❌'
    if pred.strip() == gt.strip():
        correct += 1
    print(f"{match} {img_name}  |  GT: {gt}  |  Pred: {pred}")

print(f"\nAccuracy: {correct}/{total} ({100*correct/total:.0f}%)")

## Step 7 — Save trained model to Google Drive

Copy the trained model to your Drive so you can download it later.

In [ ]:
import shutil

# Save model to Google Drive
drive_model_dir = '/content/drive/MyDrive/brahmi_ocr_model'
if os.path.exists(drive_model_dir):
    shutil.rmtree(drive_model_dir)

shutil.copytree('model/brahmi_trocr', drive_model_dir)
print(f"✅ Model saved to Google Drive: {drive_model_dir}")
print(f"   Files: {os.listdir(drive_model_dir)}")

In [ ]:
# Or download the model as a zip directly to your local machine
!zip -r /content/brahmi_trocr_model.zip model/brahmi_trocr/

from google.colab import files
files.download('/content/brahmi_trocr_model.zip')
print("✅ Model downloaded! Extract it into your local project's model/ folder.")

## Done! 🎉

### Next steps:
1. Download `brahmi_trocr_model.zip` and extract it into your local project's `model/` folder
2. Run inference locally:
   ```bash
   python inference/predict.py --image <your_image.png> --model_dir model/brahmi_trocr
   ```
3. Later: build the web app and add transliteration + translation